In [7]:
import requests
import json
import pandas as pd

### Initialize the dataframe
final_df = pd.DataFrame()

### API link
url = "https://api.hotel-audit.hrs.com/v1/audits/report?secretKey=3AwuZKz9nH&page=1&size=100"
parsed = json.loads(requests.get(url).text)
### Get the number of pages through -> #int(parsed['total_pages'])
for x in range(int(parsed['total_pages'])):
    url_iter = "https://api.hotel-audit.hrs.com/v1/audits/report?secretKey=3AwuZKz9nH&page="+str(x+1)+"&size=100"
    response = requests.get(url_iter)
    
    if response.status_code == 200:    
        data = response.text
        parsed = json.loads(data)
        df = pd.DataFrame(parsed.get('results'))
        final_df = final_df.append(df, ignore_index=True)
    else: 
        print("Request failed page: {} ".format(x))
final_df.shape

,id,hkey,name,created_date,updated_date,checked,link,program_name,version,terms,...,brand_id,brand_name,city_id,city,country_id,country,performance_cluster,status,type,missed
0,00005b15-b99c-444d-97bf-d3cd02f8c8db,431251,Homewood Suites by Hilton Atlanta I-85-Lawrenc...,2020-07-31T16:16:24.000Z,2020-07-31T16:16:24.000Z,"1,2,3,4,5,6,8,10,11,12,13,14,15,16,17,18,19,20...",https://www.hilton.com/en/corporate/cleanstay/,Hilton CleanStayTM,1,2,...,1537,Homewood Suites by Hilton,168172,Lawrenceville,165,USA,4.0,True,cleansafe_self_inspection,
1,00008c26-5cc5-4655-8170-7fdee1808e5d,110020,QUALITY INN BARRIE,2021-03-30T20:04:45.000Z,2021-03-30T20:04:45.000Z,"1,2,3,4,5,6,7,8,10,11,12,9,14,15,16,17,18,19,2...",https://www.choicehotels.com/en-au/about/commi...,Choice Hotels Commitment to Clean,1,2,...,1760,Quality by Choice,665022,Barrie,75,Canada,5.0,True,cleansafe_self_inspection,
2,000225aa-e50c-46c0-a3fe-65813215f837,442825,Hampton Inn - Suites Mount Pleasant,2020-07-31T16:16:24.000Z,2020-07-31T16:16:24.000Z,"1,2,3,4,5,6,8,10,11,12,13,14,15,16,17,18,19,20...",https://www.hilton.com/en/corporate/cleanstay/,Hilton CleanStayTM,1,2,...,1535,Hampton Inn by Hilton,190895,Mount Pleasant,165,USA,5.0,True,cleansafe_self_inspection,
3,0002b8e5-2d6c-4334-8aa4-5c831da5a5bf,373376,Sleep Inn Raleigh Durham Airport,2021-03-30T20:04:56.000Z,2021-03-30T20:04:56.000Z,"1,2,3,4,5,6,7,8,10,11,12,9,14,15,16,17,18,19,2...",https://www.choicehotels.com/en-au/about/commi...,Choice Hotels Commitment to Clean,1,2,...,1761,Sleep Inn by Choice,131636,DURHAM,165,USA,NaN,True,cleansafe_self_inspection,
4,00047545-725b-415b-ae11-642fefb43dca,61,Best Western Hotel Domicil,2021-01-06T15:33:26.000Z,2021-01-11T10:35:52.000Z,"1,2,3,4,5,6,7,8,10,11,12,9,14,15,16,17,18,19,2...",https://www.bestwestern.it/hotels/specialprote...,We Care Clean,1,2,...,60,Best Western,92734,Bonn,33,Germany,1.0,True,cleansafe_self_inspection,


In [3]:
from datetime import dt
final_df.to_excel('C:\\Users\\USER\\Documents\CleanAndSafe\\'+str(dt.today())+'_CleanAndSafe_list.xlsx', 
              sheet_name='CleanAndSafeHotels', 
              header=True,
              encoding='utf-8',
              index=False,
              freeze_panes=(1,0) )

In [2]:
final_df['created_date']= pd.to_datetime(final_df['created_date'])
final_df['updated_date']= pd.to_datetime(final_df['updated_date'])
harmonized_df = final_df[final_df.hkey.notnull()]
# harmonized_df = harmonized_df.loc[:, harmonized_df.columns != 'checked']

In [65]:
#harmonized_df[harmonized_df.hkey.isnull()]
print(f'Number of records having HOTEL_IDs {final_df.id[final_df.hkey.notnull()].count()}.')
print(f'Number of records with no HOTEL_IDs  {final_df.id[final_df.hkey.isna()].count()}.')

In [3]:
# harmonized_df = harmonized_df.head(10)
harmonized_df['checked_id'] = harmonized_df['checked']
harmonized_df.drop(['checked'], axis= 1, inplace = True)

In [123]:
harmonized_df.shape

In [124]:
import pyexasol
import configparser

#Location of the ini file
config = configparser.ConfigParser()
config.read('C:\\Users\\USER\\.spyder-py3\\pfxIUTO.ini')
dsn=config['pfxIUT']['dsn']
user=config['pfxIUT']['user']
pwd=config['pfxIUT']['pwd']
schema=config['pfxIUT']['schema']
# Exasol connection
connect = pyexasol.connect(dsn=dsn, user=user, password=pwd, schema=schema)
connect.execute("TRUNCATE TABLE DWHPFX.CLEAN_SAFE_HOTELS")
connect.import_from_pandas(harmonized_df, table = ('DWHPFX','CLEAN_SAFE_HOTELS'))
print("SUCCESS")

In [88]:
from importlib import import_module

named_libs = [('pandas', 'pd')] # (library_name, shorthand)
for (name, short) in named_libs:
    try:
        lib = import_module(name)
    except:
        print(sys.exc_info())
    else:
        globals()[short] = lib  

libnames = ['requests', 'json', 'pyexasol', 'configparser']
for libname in libnames:
    try:
        lib = import_module(libname)
    except:
        print(sys.exc_info())
    else:
        globals()[libname] = lib
        
### API EXTRACTION

def cleanNsafeExtract():
    try:
        ### Initialize the dataframe
        final_df = pd.DataFrame()
        ### API link
        url = "https://api.hotel-audit.hrs.com/v1/audits/report?secretKey=3AwuZKz9nH&page=1&size=100"
        parsed = json.loads(requests.get(url).text)
        ### Get the number of pages through -> #int(parsed['total_pages'])
        for x in range(int(parsed['total_pages'])):
            url_iter = "https://api.hotel-audit.hrs.com/v1/audits/report?secretKey=3AwuZKz9nH&page="+str(x+1)+"&size=100"
            response = requests.get(url_iter)
            if response.status_code == 200:    
                data = response.text
                parsed = json.loads(data)
                df = pd.DataFrame(parsed.get('results'))
                final_df = final_df.append(df, ignore_index=True)
            else: 
                print("Request failed page: {} ".format(x))
                
        final_df['created_date']= pd.to_datetime(final_df['created_date'])
        final_df['updated_date']= pd.to_datetime(final_df['updated_date'])
        harmonized_df = final_df[final_df.hkey.notnull()]
        harmonized_df = harmonized_df.loc[:, harmonized_df.columns != 'checked']
    except:
        print('Failed in function cleanNsafe - ')
    return harmonized_df, print(f'Number of records having HOTEL_IDs {final_df.id[final_df.hkey.notnull()].count()}.'), print(f'Number of records with no HOTEL_IDs  {final_df.id[final_df.hkey.isna()].count()}.')    

In [90]:
df, a, b = cleanNsafeExtract()
df

In [86]:
def cleanNsafeLoad(df): 
    try:
        #Location of the ini file
        config = configparser.ConfigParser()
        config.read('C:\\Users\\USER\\.spyder-py3\\pfxIUT.ini')
        dsn=config['pfxIUT']['dsn']
        user=config['pfxIUT']['user']
        pwd=config['pfxIUT']['pwd']
        schema=config['pfxIUT']['schema']
        # Exasol connection
        connect = pyexasol.connect(dsn=dsn, user=user, password=pwd, schema=schema)
        connect.execute("TRUNCATE TABLE DWHPFX.CLEAN_SAFE_HOTELS")
        connect.import_from_pandas(harmonized_df, table = ('DWHPFX','CLEAN_SAFE_HOTELS'))
        print("SUCCESS")
    except:
        print('Failed in function cleanNsafeLoad - check DB connection')
    return print(f'Number of records inserted  {harmonized_df.id[harmonized_df.hkey.notnull()].count()}.')

In [87]:
cleanNsafeLoad(harmonized_df)

In [79]:
from urllib.parse import urlsplit
parsed = urlsplit("https://api.hotel-audit.hrs.com/v1/audits/report?secretKey=3AwuZKz9nH&page=1&size=100")
print('query  :', parsed.query)

In [80]:
print('query  :', parsed.fragment)

In [148]:
test = harmonized_df[harmonized_df.missed != ""].reset_index()
test.head()

In [97]:
#test['missed_desc'] = test['missed'].replace({1:'Public health authorities guidelines', 2:'Risk assessment', 36:'Dish washing machine'})
test['missed_desc'] = test['missed'].str.replace('1','Public health authorities guidelines')
test['missed_desc'] = test['missed'].str.replace('2', 'Risk assessment')
#test['missed_desc'] = test['missed'].str.replace('36','Dish washing machine')
test['missed_desc']

In [77]:
for row in test.itertuples(name='missed'): 
    #print(row.missed) ##str
    temp = row.missed.split(',')
    print(temp)  ## list with string values

In [149]:
test = test.head(5)
test

In [138]:
missed_desc = []
for row in test.itertuples(name='missed'): 
    #print(row.missed) ##str
    temp = row.missed.split(',')
    #print(temp)  ## list with string values
    for i in range(len(temp)):
        #temp_desc = temp.replace({"1":"Public health authorities guidelines", "2":"Risk assessment"}).copy()
        if temp[i] == '1':
            temp[i] = "Public health authorities guidelines"
        elif temp[i] == '2':
            temp[i] = "Risk assessment"
        elif temp[i] == '3':
            temp[i] = "Risk assessment - hotel staff"
        elif temp[i] == '4':
            temp[i] = "Action plan"
        elif temp[i] == '5':
            temp[i] = "Log books"
        elif temp[i] == '6':
            temp[i] = "Staff/Guests well informed"
        elif temp[i] == '7':
            temp[i] = "Staff/Guests assessed"
        elif temp[i] == '8':
            temp[i] = "Pictograms"
        elif temp[i] == '9':
            temp[i] = "Updated contact info"
        elif temp[i] == '10':
            temp[i] = "Staff training"
        elif temp[i] == '11':
            temp[i] = "Tracing system"
        elif temp[i] == '12':
            temp[i] = "Hygiene - suppliers/contractors"
        elif temp[i] == '13':
            temp[i] = "Web checkin"
        elif temp[i] == '14':
            temp[i] = "Hotel hygiene - guests"
        elif temp[i] == '15':
            temp[i] = "Common area- social distancing"
        elif temp[i] == '16':
            temp[i] = "PPE/kits - staff"
        elif temp[i] == '17':
            temp[i] = "PPE/kits - guests"
        elif temp[i] == '18':
            temp[i] = "Staff remind guests"
        elif temp[i] == '19':
            temp[i] = "Disinfectant - room keys"
        elif temp[i] == '20':
            temp[i] = "Disinfected pools"
        elif temp[i] == '21':
            temp[i] = "Lockers - Spa/fitness social distancing"
        elif temp[i] == '22':
            temp[i] = "Lockers - Spa/fitness"
        elif temp[i] == '23':
            temp[i] = "Disinfectant products - fitness"
        elif temp[i] == '24':
            temp[i] = "Disinfectant reminder - fitness"
        elif temp[i] == '25':
            temp[i] = "Fitness equipment"
        elif temp[i] == '26':
            temp[i] = "Social distancing - Spa/fitness"
        elif temp[i] == '27':
            temp[i] = "Dish washers and washing machine"
        elif temp[i] == '28':
            temp[i] = "HVAC systems"
        elif temp[i] == '29':
            temp[i] = "Sanitiser dispensers"
        elif temp[i] == '30':
            temp[i] = "Public restrooms"
        elif temp[i] == '31':
            temp[i] = "Take away/Room service"
        elif temp[i] == '32':
            temp[i] = "Safety measures - buffets"
        elif temp[i] == '33':
            temp[i] = "Disinfection - buffet areas"
        elif temp[i] == '34':
            temp[i] = "Disinfection - vending machines"
        elif temp[i] == '35':
            temp[i] = "Dishwashing machine"
        elif temp[i] == '36':
            temp[i] = "Social distancing dining"
        elif temp[i] == '37':
            temp[i] = "Social distancing seating"
        elif temp[i] == '38':
            temp[i] = "Guest reminder"
        elif temp[i] == '39':
            temp[i] = "Cleaning protocol - public areas"
        elif temp[i] == '40':
            temp[i] = "Disinfection protocol - COVID cases"
        elif temp[i] == '41':
            temp[i] = "Disinfection - guest rooms"
        elif temp[i] == '42':
            temp[i] = "Dirty linen storage"
        elif temp[i] == '43':
            temp[i] = "Ventilation"
        elif temp[i] == '44':
            temp[i] = "Staff to access meeting room"
        elif temp[i] == '45':
            temp[i] = "Luggage store for group"
        elif temp[i] == '46':
            temp[i] = "Social distancing - meeting rooms"
    missed_desc.append(temp)
missed_desc       

In [141]:
test1 = [str(item) for item in missed_desc]
# test1
missed_df = pd.DataFrame(test1)
missed_df

In [150]:
test = test.join(missed_df)

In [151]:
test #257020, 874980, 728

In [184]:
# import datetime as dt
### Initialize the dataframe
final_df = pd.DataFrame()
### API link
url = "https://api.hotel-audit.hrs.com/v1/audits/report?secretKey=3AwuZKz9nH&page=1&size=100"
parsed = json.loads(requests.get(url).text)
### Get the number of pages through -> #int(parsed['total_pages'])
for x in range(int(parsed['total_pages'])):
    url_iter = "https://api.hotel-audit.hrs.com/v1/audits/report?secretKey=3AwuZKz9nH&page="+str(x+1)+"&size=100"
    response = requests.get(url_iter)
    if response.status_code == 200:    
        data = response.text
        parsed = json.loads(data)
        df = pd.DataFrame(parsed.get('results'))
        final_df = final_df.append(df, ignore_index=True)
    else: 
        print("Request failed page: {} ".format(x))

#         final_df['created_date']= pd.to_datetime(final_df['created_date'])
#         final_df['updated_date']= pd.to_datetime(final_df['updated_date'])
final_df['created_date']= final_df['created_date'].dt.tz_localize(None)
final_df['updated_date']= final_df['updated_date'].dt.tz_localize(None)
harmonized_df = final_df[final_df.hkey.notnull()]
#harmonized_df = harmonized_df.loc[:, harmonized_df.columns != 'checked']
missed_desc = []
for row in harmonized_df.itertuples(name='missed'): 
    #print(row.missed) ##str
    temp = row.missed.split(',')
    #print(temp)  ## list with string values
    for i in range(len(temp)):
        #temp_desc = temp.replace({"1":"Public health authorities guidelines", "2":"Risk assessment"}).copy()
        if temp[i] == '1':
            temp[i] = "Public health authorities guidelines"
        elif temp[i] == '2':
            temp[i] = "Risk assessment"
        elif temp[i] == '3':
            temp[i] = "Risk assessment - hotel staff"
        elif temp[i] == '4':
            temp[i] = "Action plan"
        elif temp[i] == '5':
            temp[i] = "Log books"
        elif temp[i] == '6':
            temp[i] = "Staff/Guests well informed"
        elif temp[i] == '7':
            temp[i] = "Staff/Guests assessed"
        elif temp[i] == '8':
            temp[i] = "Pictograms"
        elif temp[i] == '9':
            temp[i] = "Updated contact info"
        elif temp[i] == '10':
            temp[i] = "Staff training"
        elif temp[i] == '11':
            temp[i] = "Tracing system"
        elif temp[i] == '12':
            temp[i] = "Hygiene - suppliers/contractors"
        elif temp[i] == '13':
            temp[i] = "Web checkin"
        elif temp[i] == '14':
            temp[i] = "Hotel hygiene - guests"
        elif temp[i] == '15':
            temp[i] = "Common area- social distancing"
        elif temp[i] == '16':
            temp[i] = "PPE/kits - staff"
        elif temp[i] == '17':
            temp[i] = "PPE/kits - guests"
        elif temp[i] == '18':
            temp[i] = "Staff remind guests"
        elif temp[i] == '19':
            temp[i] = "Disinfectant - room keys"
        elif temp[i] == '20':
            temp[i] = "Disinfected pools"
        elif temp[i] == '21':
            temp[i] = "Lockers - Spa/fitness social distancing"
        elif temp[i] == '22':
            temp[i] = "Lockers - Spa/fitness"
        elif temp[i] == '23':
            temp[i] = "Disinfectant products - fitness"
        elif temp[i] == '24':
            temp[i] = "Disinfectant reminder - fitness"
        elif temp[i] == '25':
            temp[i] = "Fitness equipment"
        elif temp[i] == '26':
            temp[i] = "Social distancing - Spa/fitness"
        elif temp[i] == '27':
            temp[i] = "Dish washers and washing machine"
        elif temp[i] == '28':
            temp[i] = "HVAC systems"
        elif temp[i] == '29':
            temp[i] = "Sanitiser dispensers"
        elif temp[i] == '30':
            temp[i] = "Public restrooms"
        elif temp[i] == '31':
            temp[i] = "Take away/Room service"
        elif temp[i] == '32':
            temp[i] = "Safety measures - buffets"
        elif temp[i] == '33':
            temp[i] = "Disinfection - buffet areas"
        elif temp[i] == '34':
            temp[i] = "Disinfection - vending machines"
        elif temp[i] == '35':
            temp[i] = "Dishwashing machine"
        elif temp[i] == '36':
            temp[i] = "Social distancing dining"
        elif temp[i] == '37':
            temp[i] = "Social distancing seating"
        elif temp[i] == '38':
            temp[i] = "Guest reminder"
        elif temp[i] == '39':
            temp[i] = "Cleaning protocol - public areas"
        elif temp[i] == '40':
            temp[i] = "Disinfection protocol - COVID cases"
        elif temp[i] == '41':
            temp[i] = "Disinfection - guest rooms"
        elif temp[i] == '42':
            temp[i] = "Dirty linen storage"
        elif temp[i] == '43':
            temp[i] = "Ventilation"
        elif temp[i] == '44':
            temp[i] = "Staff to access meeting room"
        elif temp[i] == '45':
            temp[i] = "Luggage store for group"
        elif temp[i] == '46':
            temp[i] = "Social distancing - meeting rooms"
    missed_desc.append(temp)
missed_desc
temp = [str(item) for item in missed_desc]
missed_df = pd.DataFrame(temp)
harmonized_df = harmonized_df.join(missed_df)
harmonized_df.rename(columns={0: "missed_desc"}, inplace = True)

In [187]:
final_df['created_date'].tz_localize(None)

In [161]:
harmonized_df[harmonized_df.hkey == 257020]


In [163]:
harmonized_df.columns

In [164]:
harmonized_df.rename(columns={0: "missed_desc"}, inplace = True)

In [178]:
harmonized_df.loc[harmonized_df.missed_desc.str.len() > 4]

In [176]:
len(harmonized_df.missed_desc) > 4

In [7]:
import requests
import json
import pandas as pd

### Initialize the dataframe
final_df = pd.DataFrame()
url_iter = "https://api.hotel-audit.hrs.com/v1/audits/report?secretKey=3AwuZKz9nH&page=1&size=100"
response = requests.get(url_iter)
data = response.text
parsed = json.loads(data)
df = pd.DataFrame(parsed.get('results'))
df
# if response.status_code == 200:    
#     data = response.text
#     parsed = json.loads(data)
#     df = pd.DataFrame(parsed.get('results'))
#     final_df = final_df.append(df, ignore_index=True)
# final_df

In [8]:
parsed['total_pages']

In [9]:
### Initialize the dataframe
final_df = pd.DataFrame()
### API link
url = "https://api.hotel-audit.hrs.com/v1/audits/report?secretKey=3AwuZKz9nH&page=1&size=100"
parsed = json.loads(requests.get(url).text)
### Get the number of pages through -> #int(parsed['total_pages'])
for x in range(5):
    url_iter = "https://api.hotel-audit.hrs.com/v1/audits/report?secretKey=3AwuZKz9nH&page="+str(x+1)+"&size=100"
    response = requests.get(url_iter)
    if response.status_code == 403 or response.status_code == 200:    
        data = response.text
        parsed = json.loads(data)
        df = pd.DataFrame(parsed.get('results'))
        final_df = final_df.append(df, ignore_index=True)
    else: 
        print("Request failed page: {} ".format(x))

final_df['created_date']= pd.to_datetime(final_df['created_date'])
final_df['updated_date']= pd.to_datetime(final_df['updated_date'])
harmonized_df = final_df[final_df.hkey.notnull()]
harmonized_df